<a href="https://colab.research.google.com/github/rhodes-byu/stat-486/blob/main/notebooks/08-boosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b></b></p>

# AdaBoost vs XGBoost vs LightGBM

This notebook is for students who have never used boosting libraries before.

You will learn:
1. What AdaBoost, XGBoost, and LightGBM are doing conceptually
2. How to install and import each library
3. How to train all three on the same **real** dataset (Adult Income from UCI/OpenML)
4. How to evaluate and compare results
5. How to tune key hyperparameters without getting overwhelmed

### Suggested run order
Run cells from top to bottom once, then re-run sections 7–11 as you experiment with parameters.

> Learning goal: understand the modeling workflow first, then tune performance.

## 1) Big Picture Intuition

### AdaBoost (Adaptive Boosting)
- Builds many weak learners (often shallow trees) one after another
- Puts more weight on examples that were predicted incorrectly
- Final prediction is a weighted vote across all weak learners
- Usually easiest to explain conceptually

### XGBoost (Extreme Gradient Boosting)
- Adds trees sequentially to correct residual errors from previous trees
- Uses regularization and shrinkage to control overfitting
- Strong default performance on many tabular problems
- Very popular in competitions and industry

### LightGBM
- Also gradient boosting, but engineered for speed on large tabular data
- Uses histogram-based splitting and leaf-wise growth
- Often very fast and memory-efficient
- Can reach strong accuracy with careful tuning

### Common question
**Are they all “boosting”?** Yes. They differ mostly in optimization details, regularization options, and implementation speed.

## 2) Install (if needed)
Run this cell if your environment is missing packages. Restart the kernel after install if imports fail.

In [ ]:
# Uncomment and run if needed:
# %pip install -q scikit-learn pandas numpy matplotlib seaborn xgboost lightgbm joblib

## 3) Imports and Setup

In [ ]:
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 4) Load The Dataset: Adult Income (UCI / OpenML)

This is a classic real-world tabular classification problem:
- Predict whether annual income is `>50K` or `<=50K`
- Mixed numerical + categorical features
- Missing values and class imbalance

This makes it much more realistic than synthetic toy data.

In [ ]:
adult = fetch_openml(name="adult", version=2, as_frame=True)

df = adult.frame.copy()

# Target is strings ('<=50K', '>50K'). Convert to binary 0/1.
# 1 means >50K, 0 means <=50K.
y = (df["class"] == ">50K").astype(int)
X = df.drop(columns=["class"])

print("Dataset shape:", X.shape)
print("Target mean (positive class rate):", y.mean().round(3))
print("\nSample rows:")
display(X.head())

## 5) Train/Test Split + Preprocessing

Because this dataset has mixed feature types, we build a preprocessing pipeline:
- Numeric columns: median imputation
- Categorical columns: most-frequent imputation + one-hot encoding

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

numeric_features = X.select_dtypes(include=["number"]).columns
categorical_features = X.select_dtypes(exclude=["number"]).columns

numeric_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_preprocess, numeric_features),
    ("cat", categorical_preprocess, categorical_features),
])

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Numeric feature count:", len(numeric_features))
print("Categorical feature count:", len(categorical_features))

## 6) Baseline Setup

1. Define model pipeline
2. Fit model
3. Evaluate metrics
4. Inspect classification report

### How to read outputs
- Use **ROC-AUC** as the primary ranking metric here.
- Use **F1** to understand balance between precision and recall.
- Keep an eye on **Train_Time_Seconds** for practical model choice.

### Most important knobs
- **AdaBoost**: `n_estimators`, `learning_rate`, weak learner depth
- **XGBoost**: `n_estimators`, `learning_rate`, `max_depth`, `subsample`, `colsample_bytree`, regularization
- **LightGBM**: `n_estimators`, `learning_rate`, `num_leaves`, `max_depth`, `subsample`, `colsample_bytree`

Rule of thumb: smaller `learning_rate` often needs larger `n_estimators`.

In [ ]:
results = []
trained_pipelines = {}

def train_and_evaluate(name, pipeline, X_train, y_train, X_test, y_test):
    start = time.perf_counter()
    pipeline.fit(X_train, y_train)
    train_time = time.perf_counter() - start

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
        "Train_Time_Seconds": train_time,
    }
    results.append(row)
    trained_pipelines[name] = pipeline

    print(f"{name} metrics:")
    print(pd.DataFrame([row]))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, digits=3))

## 7) AdaBoost: Train + Evaluate

### Hyperparameters used here
- `estimator=DecisionTreeClassifier(max_depth=1)`: each weak learner is a decision stump (very simple tree)
- `n_estimators=250`: number of weak learners to combine
- `learning_rate=0.08`: shrinks each learner's contribution

### Guidance for students
- If underfitting, try higher `n_estimators` (e.g., 300–500).
- If overfitting or unstable results, lower `learning_rate`.
- Keep weak learners simple first (`max_depth` 1–2) so boosting is easier to interpret.

In [ ]:
adaboost_pipeline = Pipeline([
    ("prep", preprocessor),
    (
        "model",
        AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
            n_estimators=250,
            learning_rate=0.08,
            random_state=RANDOM_STATE,
        ),
    ),
])

train_and_evaluate(
    name="AdaBoost",
    pipeline=adaboost_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
)

## 8) XGBoost: Train + Evaluate

### Hyperparameters used here
- `n_estimators=300`: number of boosting rounds (trees)
- `learning_rate=0.05`: step size; smaller values often generalize better
- `max_depth=6`: maximum depth of each tree
- `min_child_weight=1`: minimum instance weight in a child node
- `subsample=0.85`: row subsampling per tree
- `colsample_bytree=0.85`: feature subsampling per tree
- `reg_alpha=0.0`, `reg_lambda=1.0`: L1/L2 regularization

### Guidance for students
- Start by tuning `learning_rate`, `n_estimators`, and `max_depth` first.
- If overfitting, reduce `max_depth` and/or lower `subsample`/`colsample_bytree`.
- If underfitting, increase `n_estimators` or slightly increase depth.

In [ ]:
# 8) XGBoost: Train + Evaluate
xgboost_pipeline = Pipeline([
    ("prep", preprocessor),
    (
        "model",
        XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            min_child_weight=1,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.0,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    ),
])

train_and_evaluate(
    name="XGBoost",
    pipeline=xgboost_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
)

### Quick check (after XGBoost)
- Compare XGBoost to AdaBoost: where do metrics improve most?

## 9) LightGBM: Train + Evaluate

### Hyperparameters used here
- `n_estimators=300`: number of trees
- `learning_rate=0.05`: shrinkage per boosting step
- `num_leaves=31`: max leaves per tree (key complexity control in LightGBM)
- `max_depth=-1`: no explicit depth cap
- `subsample=0.85`: row subsampling
- `colsample_bytree=0.85`: feature subsampling

### Guidance for students
- Tune `num_leaves` and `learning_rate` first.
- Larger `num_leaves` increases model flexibility and overfitting risk.
- If results are noisy, lower `num_leaves` or set a finite `max_depth` (e.g., 6–12).

In [ ]:
lightgbm_pipeline = Pipeline([
    ("prep", preprocessor),
    (
        "model",
        LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        ),
    ),
])

train_and_evaluate(
    name="LightGBM",
    pipeline=lightgbm_pipeline,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
)

### Quick check (after LightGBM)
- How does LightGBM compare in speed vs ROC-AUC?

## 10) Baseline Summary and Visual Comparison

Now that all three models were trained separately, we combine results for a side-by-side view.

In [ ]:
results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
display(results_df)

plot_df = results_df.melt(
    id_vars="Model",
    value_vars=["Accuracy", "F1", "ROC_AUC"],
    var_name="Metric",
    value_name="Value",
)

plt.figure(figsize=(10, 5))
sns.barplot(data=plot_df, x="Metric", y="Value", hue="Model")
plt.ylim(0.0, 1.0)
plt.title("Baseline Metric Comparison")
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(data=results_df, x="Model", y="Train_Time_Seconds")
plt.title("Baseline Training Time (seconds)")
plt.tight_layout()
plt.show()

## 11) Hyperparameter Tuning Quick Start (XGBoost example)

Start simple: tune one model first, then apply the same process to others.

Key parameters to tune first:
- `n_estimators`
- `learning_rate`
- `max_depth`
- `subsample`
- `colsample_bytree`
- regularization (`reg_lambda`)

In [ ]:
xgb_tune_pipeline = Pipeline([
    ("prep", preprocessor),
    (
        "model",
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    ),
])

param_distributions = {
    "model__n_estimators": [150, 250, 350, 500],
    "model__learning_rate": [0.02, 0.05, 0.1],
    "model__max_depth": [3, 4, 6, 8],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.7, 0.85, 1.0],
    "model__reg_lambda": [0.5, 1.0, 2.0],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

xgb_search = RandomizedSearchCV(
    estimator=xgb_tune_pipeline,
    param_distributions=param_distributions,
    n_iter=10,
    scoring="roc_auc",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

search_start = time.perf_counter()
xgb_search.fit(X_train, y_train)
search_time = time.perf_counter() - search_start

best_xgb = xgb_search.best_estimator_
best_xgb_pred = best_xgb.predict(X_test)
best_xgb_proba = best_xgb.predict_proba(X_test)[:, 1]

print("Best params:", xgb_search.best_params_)
print(f"Search time (s): {search_time:.2f}")
print(f"Tuned XGBoost ROC-AUC: {roc_auc_score(y_test, best_xgb_proba):.4f}")
print(f"Tuned XGBoost F1: {f1_score(y_test, best_xgb_pred):.4f}")

## 12) Feature Importance Snapshot

Feature importance scales differ across frameworks, so compare rankings within each model more than absolute values across models.

In [ ]:
top_k = 12
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (name, pipeline) in zip(axes, trained_pipelines.items()):
    prep = pipeline.named_steps["prep"]
    model = pipeline.named_steps["model"]

    feature_names = prep.get_feature_names_out()
    importances = pd.Series(model.feature_importances_, index=feature_names)
    top_features = importances.sort_values(ascending=False).head(top_k)

    sns.barplot(x=top_features.values, y=top_features.index, ax=ax)
    ax.set_title(f"{name}: Top {top_k} Features")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")

plt.tight_layout()
plt.show()

## 13) Practical Recommendations for Students

- Start with a clean preprocessing pipeline before tuning any model.
- Build baseline metrics first; tuning without a baseline is hard to interpret.
- Tune 2–4 high-impact hyperparameters first, not every parameter at once.
- Compare both predictive performance **and** training time.
- Keep train/test split and random seed fixed while comparing models.

Suggested exercises:
1. Repeat the same tuning procedure for LightGBM.
2. Tune AdaBoost (`n_estimators`, `learning_rate`, weak learner depth).
3. Use cross-validation folds = 5 and compare metric stability.